# Connectome Reconstruction Error Analysis: Hypothesis-Testing Subsystem

**Hypothesis Testing: Real Connectome vs. Matched Random/Null Connectome**

Tests whether secondary structural effects (edge buffering, reciprocity shifts, component fragmentation, PageRank stability) differ significantly between real biological connectomes and randomized degree/weight-matched null graphs.

---
### How to run on Kaggle / Locally
1. In **Cell 3**, set `DATASET_NAME`, `RUN_REAL`, `RUN_NULL`, `ERROR_MODELS`, `ERROR_RATES`, and `RANDOM_SEEDS`.
2. Click **Run All**.
3. Review the statistical comparison tables, FDR-corrected p-values, and narrative interpretations generated at the bottom.

In [ ]:
# Cell 1: Environment Setup & sys.path
# ============================================================
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

KAGGLE_CODEBASE_PATH = Path('/kaggle/input/datasets/jeet7771/flywire-codebase')
KAGGLE_DATA_PATH     = Path('/kaggle/input/datasets/jeet7771/flywire-all-datasets')

if IS_KAGGLE:
    if not KAGGLE_CODEBASE_PATH.exists():
        raise FileNotFoundError(f'Codebase not found at {KAGGLE_CODEBASE_PATH}')
    sys.path.insert(0, str(KAGGLE_CODEBASE_PATH))
    print(f'[OK] Codebase path: {KAGGLE_CODEBASE_PATH}')
    if not KAGGLE_DATA_PATH.exists():
        raise FileNotFoundError(f'Data not found at {KAGGLE_DATA_PATH}')
    print(f'[OK] Data path    : {KAGGLE_DATA_PATH}')
else:
    REPO_ROOT = Path(os.getcwd()).resolve()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    print(f'[OK] Running locally. Repo root: {REPO_ROOT}')

print(f'Environment: {"KAGGLE" if IS_KAGGLE else "LOCAL"}')

In [ ]:
# ============================================================
# Cell 3: RUNTIME CONFIGURATION  <-- EDIT THIS CELL AS NEEDED
# ============================================================

# Execution mode:
#   - 'null_only'        : Runs ONLY on Null graphs (bypasses duplicate Real runs, ~12-15m)
#   - 'full'             : Runs on BOTH Real connectome and Null graphs (~25-35m)
#   - 'compare_existing': Lightweight comparison of existing Real results vs Null results (<1s)
EXECUTION_MODE = 'null_only'

# Target connectome dataset: 'BANC' | 'FAFB' | 'MANC' | 'MAOL' | 'MCNS' | 'TEST'
DATASET_NAME = 'BANC'

# [LOCAL ONLY] Path to raw data folder
LOCAL_DATASET_ROOT = 'research_data/raw'

# Null model type: 'degree_preserving' (primary) or 'erdos_renyi'
NULL_MODEL_NAME = 'degree_preserving'

# Number of independently generated null graphs (set to [1] for single null graph):
NULL_GRAPH_SEEDS = [1]

# Error models to evaluate
ERROR_MODELS = [
    'missed_synapses',
    'false_synapses',
    'synapse_count_measurement',
    'split_errors',
    'merge_errors',
]

# --- ERROR RATES ---
ERROR_RATES = [0.000, 0.005, 0.010, 0.020, 0.030, 0.050, 0.075, 0.100, 0.150, 0.200]

# --- PERTURBATION TRIAL SEEDS ---
# Random seeds controlling error-model stochasticity on each null graph:
RANDOM_SEEDS = [1, 2, 3, 4, 5]

# Graph analyses to execute
ANALYSES = [
    'basic_structure',
    'degree_distribution',
    'connected_components',
    'reciprocity',
    'pagerank',
]

# Output directory root
OUTPUT_ROOT = Path('results') / 'hypothesis_testing'

print(f'Execution Mode    : {EXECUTION_MODE}')
print(f'Dataset Name      : {DATASET_NAME}')
print(f'Null Model        : {NULL_MODEL_NAME}')
print(f'Null Graph Seeds  : {NULL_GRAPH_SEEDS} ({len(NULL_GRAPH_SEEDS)} graph topology)')
print(f'Perturbation Seeds: {RANDOM_SEEDS} ({len(RANDOM_SEEDS)} trials per model/rate)')
print(f'Error Models      : {ERROR_MODELS}')
print(f'Error Rates       : {ERROR_RATES}')
print(f'Output Root       : {OUTPUT_ROOT}')

In [ ]:
# Cell 4: Resolve Dataset Path & Assemble Config
# ============================================================
if IS_KAGGLE:
    DATASET_ROOT = str(KAGGLE_DATA_PATH)
    CONFIGS_ROOT = str(KAGGLE_CODEBASE_PATH / 'configs')
else:
    DATASET_ROOT = '0-demodata' if DATASET_NAME.upper() == 'TEST' else LOCAL_DATASET_ROOT
    CONFIGS_ROOT = 'configs'

exp_config = HypothesisExperimentConfig(
    dataset_name=DATASET_NAME,
    dataset_root=DATASET_ROOT,
    configs_root=CONFIGS_ROOT,
    execution_mode=EXECUTION_MODE,
    null_model_name=NULL_MODEL_NAME,
    null_graph_seeds=NULL_GRAPH_SEEDS,
    error_model_names=ERROR_MODELS,
    error_rates=ERROR_RATES,
    random_seeds=RANDOM_SEEDS,
    analysis_names=ANALYSES,
    output_root=str(OUTPUT_ROOT),
)

print(f'[OK] Experiment configuration prepared for mode: {exp_config.execution_mode.value}.')

In [ ]:
# Cell 4: Resolve Dataset Path & Assemble Config
# ============================================================
if IS_KAGGLE:
    DATASET_ROOT = str(KAGGLE_DATA_PATH)
    CONFIGS_ROOT = str(KAGGLE_CODEBASE_PATH / 'configs')
else:
    DATASET_ROOT = '0-demodata' if DATASET_NAME.upper() == 'TEST' else LOCAL_DATASET_ROOT
    CONFIGS_ROOT = 'configs'

exp_config = HypothesisExperimentConfig(
    dataset_name=DATASET_NAME,
    dataset_root=DATASET_ROOT,
    configs_root=CONFIGS_ROOT,
    run_real=RUN_REAL,
    run_null=RUN_NULL,
    null_model_name=NULL_MODEL_NAME,
    error_model_names=ERROR_MODELS,
    error_rates=ERROR_RATES,
    random_seeds=RANDOM_SEEDS,
    analysis_names=ANALYSES,
    output_root=str(OUTPUT_ROOT),
)

print('[OK] Experiment configuration prepared.')

In [ ]:
# Cell 5: Execute Hypothesis Testing Pipeline
# ============================================================
runner = HypothesisExperimentRunner()
result = runner.run(exp_config)

print(f'Execution Status: {result.status}')
print(f'Total Runtime   : {result.runtime_seconds:.2f} seconds')
print(f'Deliverables    : {list(result.exported_paths.keys())}')

In [ ]:
# Cell 6: Display Markdown Narrative Report
# ============================================================
md_file = result.exported_paths.get('summary_markdown')
if md_file and md_file.exists():
    with open(md_file, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print('Summary report not available (both Real and Null conditions required).')

In [ ]:
# Cell 7: Visualizing Real vs. Null Secondary Effect Trajectories
# ============================================================
csv_file = result.exported_paths.get('hypothesis_test_results')
if csv_file and csv_file.exists():
    df_res = pd.read_csv(csv_file)
    
    # Filter to secondary emergent metrics only
    sec_df = df_res[df_res['category'] == 'secondary_emergent']
    unique_models = sec_df['error_model'].unique()
    
    fig, axes = plt.subplots(len(unique_models), 1, figsize=(10, 3.5 * len(unique_models)), squeeze=False)
    
    for idx, em in enumerate(unique_models):
        ax = axes[idx, 0]
        em_data = sec_df[sec_df['error_model'] == em]
        
        for metric in em_data['metric_name'].unique():
            m_data = em_data[em_data['metric_name'] == metric].sort_values('error_rate')
            rates = m_data['error_rate'] * 100
            
            ax.plot(rates, m_data['real_mean_effect'] * 100, marker='o', label=f'{metric} (Real)')
            ax.plot(rates, m_data['null_mean_effect'] * 100, marker='s', linestyle='--', label=f'{metric} (Null)')
            
        ax.axhline(0, color='gray', linestyle=':', alpha=0.6)
        ax.set_title(f'Secondary Effect Response: {em}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Error Rate (%)')
        ax.set_ylabel('Relative Change vs. Baseline (%)')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, alpha=0.3)
        
    plt.tight_layout()
    plt.show()
else:
    print('Comparison results not available for plotting.')